The goal of Step 5 is to determine whether a pretrained CNN can generalize better than our CNN trained from scratch.

We'll eventually compare the models using:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix
- Training time
- Model size
- Per-class performance

### 1. Import libraries

In [43]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import ResNet50

from tensorflow.keras.applications.mobilenet_v2 import (
    preprocess_input as mobilenet_preprocess
)

from tensorflow.keras.applications.resnet50 import (
    preprocess_input as resnet_preprocess
)

from sklearn.metrics import classification_report, confusion_matrix

In [44]:
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [45]:
print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

GPU: []


### 2. Load CSV files

In [46]:
PROJECT_ROOT = Path('..')
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [47]:
train_df = pd.read_csv(PROCESSED_DIR/'train.csv')
val_df = pd.read_csv(PROCESSED_DIR /'val.csv')
test_df = pd.read_csv(PROCESSED_DIR /'test.csv')

In [48]:
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

Training: 4480
Validation: 1120
Testing: 1600


### 3. Define classes

In [49]:
CLASSES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary"
]

NUM_CLASSES = len(CLASSES)

IMG_SIZE = 224
BATCH_SIZE = 32

### 4. Create MobileNetV2 preprocessing

In [50]:
def preprocess_mobilenet(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([
        None,
        None,
        3
    ])

    image = tf.image.resize_with_pad(
        image,
        IMG_SIZE,
        IMG_SIZE
    )

    image = tf.cast(
        image,
        tf.float32
        )

    image = preprocess_input(image)

    return image, label

preprocess_input() converts the image into the representation expected by MobileNetV2.

### 5. Create datasets

In [51]:
def create_mobilenet_dataset(df, shuffle=False):
    paths = df['filepath'].values
    labels = df['label'].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        preprocess_mobilenet,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=42
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [52]:
train_dataset = create_mobilenet_dataset(
    train_df,
    shuffle=True
)

validation_dataset = create_mobilenet_dataset(
    val_df,
    shuffle=True
)

test_datasest = create_mobilenet_dataset(
    test_df,
    shuffle=True
)

### 6. Verify preprocessing

In [53]:
images, labels = next(
    iter(train_dataset)
)

print("Shape:", images.shape)
print("Labels:", labels.shape)
print("Minimum:", tf.reduce_min(images).numpy())
print("Maximum:", tf.reduce_max(images).numpy())

Shape: (32, 224, 224, 3)
Labels: (32,)
Minimum: -123.68
Maximum: 151.061


### 7. Load pretrained MobileNetV2

In [54]:
base_model = MobileNetV2(
    weights = 'imagenet',
    include_top = False,
    input_shape = (224,224,3)
)

weights="imagenet" means we're using pretrained weights instead of starting from random weights.

include_top=False removes MobileNetV2's original ImageNet classification layer.

### 8. Freeze the pretrained layers

In [55]:
base_model.trainable = False

This means:
```text
MobileNetV2
│
├── Pretrained layers → FROZEN
│
└── New classification head → TRAINABLE

### 9. Build the transfer-learning model

In [56]:
inputs = layers.Input(
    shape=(224, 224, 3)
)

x = base_model(
    inputs,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation='softmax'
)(x)

model = models.Model(
    inputs,
    outputs
)

### 10. View the architecture

In [57]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │         5,124 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 5,124 (20.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Because the base MobileNetV2 is frozen, most parameters should be Non-trainable. while only the final classification layer is trainable.

### 11. Compile

In [58]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate = 0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']   
)

### 12. Train the frozen model

In [59]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [60]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[
        early_stopping
    ]
)

Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 104s 713ms/step - accuracy: 0.6708 - loss: 0.8041 - val_accuracy: 0.8250 - val_loss: 0.5124
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 138s 940ms/step - accuracy: 0.8105 - loss: 0.5042 - val_accuracy: 0.8429 - val_loss: 0.4216
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 122s 818ms/step - accuracy: 0.8395 - loss: 0.4293 - val_accuracy: 0.8500 - val_loss: 0.3858
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 140s 814ms/step - accuracy: 0.8585 - loss: 0.3787 - val_accuracy: 0.8732 - val_loss: 0.3603
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 120s 808ms/step - accuracy: 0.8663 - loss: 0.3603 - val_accuracy: 0.8696 - val_loss: 0.3581
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 122s 828ms/step - accuracy: 0.8833 - loss: 0.3372 - val_accuracy: 0.8830 - val_loss: 0.3326
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 113s 760ms/step - accuracy: 0.8808 - loss: 0.3248 - val_accuracy: 0.8839 - val_loss: 0.3270
Epoch 8/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 83s 582ms/step - accuracy: 0.8821 - 

### 13. Save the first model

In [61]:
MODEL_DIR = (PROJECT_ROOT/"models"/"classification")
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [62]:
model.save(
    MODEL_DIR /
    "mobilenetv2_frozen.keras"
)

### 14. Evaluate validation performance

In [63]:
val_loss, val_accuracy = model.evaluate(validation_dataset, verbose=1)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 208ms/step - accuracy: 0.8929 - loss: 0.2879
Validation Loss: 0.2878740727901459
Validation Accuracy: 0.8928571343421936


### 15. Important: test evaluation

We eventually need the test result, but I recommend not repeatedly tuning based on the test set.

In [64]:
test_loss, test_accuracy = model.evaluate(test_datasest, verbose=1)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 233ms/step - accuracy: 0.8494 - loss: 0.5878
Test Loss: 0.5877586603164673
Test Accuracy: 0.8493750095367432


Therefore the final MobileNetV2 result, because we haven't fine-tuned it.

# Fine-tune MobileNetV2

### 1. Unfreeze the MobileNetV2 base

In [65]:
base_model.trainable = True

In [66]:
print("Total layers:", len(base_model.layers))

Total layers: 154


For the first fine-tuning experiment, unfreezing only the last ~30 layers.

In [67]:
fine_tune_at = len(base_model.layers) - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = False

### 2. Keep BatchNormalization frozen

In [68]:
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

### 3. Recompile with a very small learning rate

Do not use 0.001 for fine-tuning.
That's too aggressive for changing pretrained weights.

In [69]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### 4. Train the fine-tuned model

In [70]:
fine_tune_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

In [71]:
fine_tune_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[
        fine_tune_early_stopping
    ]
)

Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 43s 282ms/step - accuracy: 0.9094 - loss: 0.2399 - val_accuracy: 0.8982 - val_loss: 0.2829
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 41s 286ms/step - accuracy: 0.9098 - loss: 0.2437 - val_accuracy: 0.8946 - val_loss: 0.2811
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 98s 691ms/step - accuracy: 0.9085 - loss: 0.2458 - val_accuracy: 0.8929 - val_loss: 0.2803
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 151s 712ms/step - accuracy: 0.9087 - loss: 0.2508 - val_accuracy: 0.8929 - val_loss: 0.2799
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 498ms/step - accuracy: 0.9080 - loss: 0.2465 - val_accuracy: 0.8929 - val_loss: 0.2799
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 42s 288ms/step - accuracy: 0.9138 - loss: 0.2452 - val_accuracy: 0.8938 - val_loss: 0.2800
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 106s 749ms/step - accuracy: 0.9138 - loss: 0.2456 - val_accuracy: 0.8938 - val_loss: 0.2797
Epoch 8/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 82s 539ms/step - accuracy: 0.9105 - loss:

### 5. Evaluate validation

In [72]:
val_loss, val_accuracy = model.evaluate(
    validation_dataset,
    verbose=1
)

print("Fine-tuned Validation Loss:", val_loss)
print("Fine-tuned Validation Accuracy:", val_accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 24s 623ms/step - accuracy: 0.8938 - loss: 0.2793
Fine-tuned Validation Loss: 0.2793136239051819
Fine-tuned Validation Accuracy: 0.893750011920929


### 6. evaluate the test set

In [73]:
test_loss, test_accuracy = model.evaluate(
    test_datasest,
    verbose=1
)

print("Fine-tuned Test Loss:", test_loss)
print("Fine-tuned Test Accuracy:", test_accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 33s 616ms/step - accuracy: 0.8438 - loss: 0.6090
Fine-tuned Test Loss: 0.6089681386947632
Fine-tuned Test Accuracy: 0.84375


Fine-tuning has not produced a meaningful improvement over the frozen model so far.

### 7. Save the first model

In [74]:
MODEL_DIR = (PROJECT_ROOT/"models"/"classification")

model.save(
    MODEL_DIR /
    "mobilenetv2_Fine_tuned.keras"
)

# ResNet50

### 1. ResNet50 preprocessing

In [75]:
def preprocess_resnet(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize_with_pad(
        image,
        IMG_SIZE,
        IMG_SIZE
    )

    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)

    return image, label

### 2. Create dataset

In [76]:
def create_resnet_dataset(df, shuffle=False):
    paths = df["filepath"].values
    labels = df["label"].values

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    dataset = dataset.map(
        preprocess_resnet,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=42
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [77]:
train_resnet = create_resnet_dataset(
    train_df,
    shuffle=True
)

val_resnet = create_resnet_dataset(
    val_df
)

test_resnet = create_resnet_dataset(
    test_df
)

In [78]:
images, labels = next(iter(train_resnet))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Pixel minimum:", tf.reduce_min(images).numpy())
print("Pixel maximum:", tf.reduce_max(images).numpy())

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Pixel minimum: -123.68
Pixel maximum: 151.061


The pixel values will not be 0–1 because ResNet50's preprocess_input() performs its own preprocessing.

###  3. Build ResNet50

Start with the base model frozen:

In [79]:
base_resnet = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_resnet.trainable = False

Create the classification head:

In [80]:
inputs = layers.Input(
    shape = (224, 224, 3)
)

x = base_resnet(
    inputs,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

resnet_model = models.Model(
    inputs,
    outputs
)

### 4. Compile

In [81]:
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate = 0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [82]:
resnet_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 8,196 (32.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

### 5. Train frozen ResNet50

In [83]:
resnet_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [84]:
resnet_history = resnet_model.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=15,
    callbacks=[resnet_early_stopping]
)

Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 483s 3s/step - accuracy: 0.7603 - loss: 0.6007 - val_accuracy: 0.8750 - val_loss: 0.3359
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 398s 3s/step - accuracy: 0.8650 - loss: 0.3637 - val_accuracy: 0.9018 - val_loss: 0.2753
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 390s 3s/step - accuracy: 0.8888 - loss: 0.2915 - val_accuracy: 0.9161 - val_loss: 0.2478
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 235s 2s/step - accuracy: 0.9058 - loss: 0.2455 - val_accuracy: 0.9107 - val_loss: 0.2432
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 126s 893ms/step - accuracy: 0.9205 - loss: 0.2234 - val_accuracy: 0.9116 - val_loss: 0.2359
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 128s 903ms/step - accuracy: 0.9176 - loss: 0.2197 - val_accuracy: 0.8964 - val_loss: 0.2664
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 128s 903ms/step - accuracy: 0.9205 - loss: 0.2050 - val_accuracy: 0.9196 - val_loss: 0.2239
Epoch 8/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 128s 903ms/step - accuracy: 0.9288 - loss: 0.195

### 6. Evaluate validation first

In [85]:
resnet_val_loss, resnet_val_accuracy = resnet_model.evaluate(
    val_resnet,
    verbose=1
)

print("ResNet50 Validation Loss:", resnet_val_loss)
print("ResNet50 Validation Accuracy:", resnet_val_accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 26s 739ms/step - accuracy: 0.9330 - loss: 0.1743
ResNet50 Validation Loss: 0.1742899864912033
ResNet50 Validation Accuracy: 0.9330357313156128


Then evaluate the test set only after training is completely finished:

In [86]:
resnet_test_loss, resnet_test_accuracy = resnet_model.evaluate(
    test_resnet,
    verbose=1
)

print("ResNet50 Test Loss:", resnet_test_loss)
print("ResNet50 Test Accuracy:", resnet_test_accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 37s 738ms/step - accuracy: 0.8888 - loss: 0.4584
ResNet50 Test Loss: 0.458353191614151
ResNet50 Test Accuracy: 0.8887500166893005


### 7. Save model 

In [87]:
MODEL_DIR = PROJECT_ROOT / "models" / "classification"

resnet_frozen_path = MODEL_DIR / "resnet50_frozen.keras"

resnet_model.save(resnet_frozen_path)

print(f"Saved: {resnet_frozen_path}")

Saved: ..\models\classification\resnet50_frozen.keras


# Fine-tune ResNet50

In [88]:
base_resnet.trainable = True

fine_tune_at = len(base_resnet.layers) - 30

for layer in base_resnet.layers[:fine_tune_at]:
    layer.trainable = False

for layer in base_resnet.layers[fine_tune_at:]:
    layer.trainable = True

# Keep BatchNorm frozen
for layer in base_resnet.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

In [89]:
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [90]:
resnet_fine_tune_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [91]:
resnet_fine_history = resnet_model.fit(
    train_resnet,
    validation_data=val_resnet,
    epochs=15,
    callbacks=[resnet_fine_tune_early_stopping]
)

Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.9471 - loss: 0.1384 - val_accuracy: 0.9536 - val_loss: 0.1371
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 157s 1s/step - accuracy: 0.9763 - loss: 0.0729 - val_accuracy: 0.9500 - val_loss: 0.1335
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 158s 1s/step - accuracy: 0.9857 - loss: 0.0443 - val_accuracy: 0.9491 - val_loss: 0.1333
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 160s 1s/step - accuracy: 0.9935 - loss: 0.0231 - val_accuracy: 0.9643 - val_loss: 0.1163
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 158s 1s/step - accuracy: 0.9953 - loss: 0.0174 - val_accuracy: 0.9607 - val_loss: 0.1231
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.9982 - loss: 0.0086 - val_accuracy: 0.9598 - val_loss: 0.1321
Epoch 7/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.9967 - loss: 0.0101 - val_accuracy: 0.9589 - val_loss: 0.1389
Epoch 8/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 160s 1s/step - accuracy: 0.9993 - loss: 0.0056 - val_accu

In [92]:
resnet_ft_test_loss, resnet_ft_test_accuracy = resnet_model.evaluate(
    test_resnet,
    verbose=1
)

print("Fine-tuned ResNet50 Test Loss:", resnet_ft_test_loss)
print("Fine-tuned ResNet50 Test Accuracy:", resnet_ft_test_accuracy)

50/50 ━━━━━━━━━━━━━━━━━━━━ 38s 751ms/step - accuracy: 0.9169 - loss: 0.5562
Fine-tuned ResNet50 Test Loss: 0.5561986565589905
Fine-tuned ResNet50 Test Accuracy: 0.9168750047683716


In [93]:
resnet_ft_val_loss, resnet_ft_val_accuracy = resnet_model.evaluate(
    val_resnet,
    verbose=1
)

print("ResNet50 Validation Loss:", resnet_ft_val_loss)
print("ResNet50 Validation Accuracy:", resnet_ft_val_accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 26s 740ms/step - accuracy: 0.9643 - loss: 0.1163
ResNet50 Validation Loss: 0.11628349870443344
ResNet50 Validation Accuracy: 0.9642857313156128


In [94]:
MODEL_DIR = PROJECT_ROOT / "models" / "classification"

resnet_ft_path = MODEL_DIR / "resnet50_finetuned.keras"

resnet_model.save(resnet_ft_path)

print(f"Saved: {resnet_ft_path}")

Saved: ..\models\classification\resnet50_finetuned.keras
